<a href="https://colab.research.google.com/github/Sanugiw/ml-fastvit/blob/Imesh/fastvit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Sanugiw/ml-fastvit.git
%cd ml-fastvit

Cloning into 'ml-fastvit'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 75 (delta 23), reused 27 (delta 12), pack-reused 29 (from 1)
Receiving objects: 100% (75/75), 3.14 MiB | 22.35 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/ml-fastvit


In [ ]:
!pip install -r requirements.txt

In [2]:
%cd /content/ml-fastvit

import os
import torch

print("Files:", os.listdir("."))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/ml-fastvit
Files: ['docs', 'models', 'export_model.py', 'requirements.txt', 'LICENSE', '.gitignore', 'train.py', 'README.md', 'misc', '.git', 'validate.py', 'ACKNOWLEDGEMENTS']
PyTorch: 2.14.1+cpu
CUDA available: False


In [ ]:
!cat requirements.txt

timm
scipy


In [ ]:
!ls -la

total 164
drwxr-xr-x 6 root root  4096 Nov 23 08:42 .
drwxr-xr-x 1 root root  4096 Nov 23 08:42 ..
-rw-r--r-- 1 root root 37036 Nov 23 08:42 ACKNOWLEDGEMENTS
drwxr-xr-x 4 root root  4096 Nov 23 08:42 docs
-rw-r--r-- 1 root root  2479 Nov 23 08:42 export_model.py
drwxr-xr-x 8 root root  4096 Nov 23 08:42 .git
-rw-r--r-- 1 root root  1822 Nov 23 08:42 .gitignore
-rw-r--r-- 1 root root  2666 Nov 23 08:42 LICENSE
drwxr-xr-x 2 root root  4096 Nov 23 08:42 misc
drwxr-xr-x 3 root root  4096 Nov 23 08:42 models
-rw-r--r-- 1 root root 15446 Nov 23 08:42 README.md
-rw-r--r-- 1 root root    11 Nov 23 08:42 requirements.txt
-rw-r--r-- 1 root root 46744 Nov 23 08:42 train.py
-rw-r--r-- 1 root root 16610 Nov 23 08:42 validate.py


In [3]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# List the contents of the main My Drive directory to locate the dataset folder
drive_path = '/content/drive/MyDrive'
if os.path.exists(drive_path):
    print("Drive Contents:", os.listdir(drive_path))
else:
    print("Google Drive is not accessible.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive Contents: ['Colab Notebooks', 'Getting started.pdf', 'Classroom', 'Imesh Mareesha - online Q 3 gc.docx', 'Imesh Mareesha - online Q 2 GC.docx', 'EA20BAAE-22BD-47CC-A1F4-4072C098450A.jpeg', 'e08f90', 'IMG_2113 (1).HEIC', 'IMG_2114 (1).HEIC', 'IMG_2121 (1).HEIC', 'IMG_2122 (1).HEIC', 'IMG_2117 (1).HEIC', 'IMG_2119 (1).HEIC', 'IMG_2116 (1).HEIC', 'IMG_2120 (1).HEIC', 'IMG_2118 (1).HEIC', 'IMG_2115 (1).HEIC', 'IMG_2123 (1).HEIC', 'IMG_2124 (1).HEIC', 'IMG_2133 (1).HEIC', 'IMG_2134 (1).HEIC', 'IMG_2127 (1).HEIC', 'IMG_2129 (1).HEIC', 'IMG_2130 (1).HEIC', 'IMG_2128 (1).HEIC', 'IMG_2132 (1).HEIC', 'IMG_2113.HEIC', 'IMG_2121.HEIC', 'IMG_2114.HEIC', 'IMG_2122.HEIC', 'IMG_2116.HEIC', 'IMG_2115.HEIC', 'IMG_2120.HEIC', 'IMG_2117.HEIC', 'IMG_2118.HEIC', 'IMG_2119.HEIC', 'IMG_2123.HEIC', 'IMG_2124.HEIC', 'IMG_2133.HEIC', 'IMG_2125.HEIC', 'IMG_2126.HEIC', 'IMG_2134.HE

In [ ]:
import os
import shutil
import zipfile
from sklearn.model_selection import train_test_split

# Define target structured directory
dataset_root = "/content/imagenet-1k"
os.makedirs(dataset_root, exist_ok=True)

# Define helper to organize a flat or class-based folder into ImageNet-1k format
def structure_dataset(source_dir, train_ratio=0.8):
    """
    Collects all images from source_dir, splits them into train/val,
    and structures them as required.
    """
    print(f"Scanning source directory: {source_dir}")
    categories = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]

    if not categories:
        print("No subdirectories found. Assuming single class or flat image folder.")
        categories = ['.']

    for category in categories:
        cat_path = os.path.join(source_dir, category) if category != '.' else source_dir
        # Filter image formats
        images = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp'))]
        if not images:
            continue

        # Resolve category name safely
        cat_name = "class_0" if category == '.' else category

        train_imgs, val_imgs = train_test_split(images, test_size=(1.0 - train_ratio), random_state=42)

        # Create directories
        os.makedirs(os.path.join(dataset_root, 'train', cat_name), exist_ok=True)
        os.makedirs(os.path.join(dataset_root, 'validation', cat_name), exist_ok=True)

        # Copy files
        for img in train_imgs:
            shutil.copy(os.path.join(cat_path, img), os.path.join(dataset_root, 'train', cat_name, img))
        for img in val_imgs:
            shutil.copy(os.path.join(cat_path, img), os.path.join(dataset_root, 'validation', cat_name, img))

    print(f"Structuring complete! Your dataset is now prepared at: {dataset_root}")

# Example usage: Set 'source_path' to point to your specific dataset folder/zip inside Google Drive
# e.g., source_path = '/content/drive/MyDrive/octwave_artifacts' or we can extract a ZIP file.
source_path = '/content/drive/MyDrive/octwave_artifacts' # Update this path to match your target dataset

if os.path.exists(source_path):
    if source_path.endswith('.zip'):
        print("Unzipping dataset to temp folder...")
        with zipfile.ZipFile(source_path, 'r') as zip_ref:
            zip_ref.extractall('/content/temp_dataset')
        structure_dataset('/content/temp_dataset')
    else:
        structure_dataset(source_path)
else:
    print(f"Specified source path '{source_path}' does not exist. Please update 'source_path' to point to your dataset directory or zip file in Drive.")

Scanning source directory: /content/drive/MyDrive/octwave_artifacts
Structuring complete! Your dataset is now prepared at: /content/imagenet-1k


In [ ]:
import os

drive_path = '/content/drive/MyDrive'

# Search Google Drive for files or folders matching 'imagenet'
matches = []
for root, dirs, files in os.walk(drive_path):
    # Limit search depth to keep it fast
    depth = root.replace(drive_path, '').count(os.sep)
    if depth > 2:
        continue
    for name in dirs + files:
        if 'imagenet' in name.lower():
            matches.append(os.path.join(root, name))

print("Found matching ImageNet files/folders on your Drive:")
for m in matches:
    print(m)

Found matching ImageNet files/folders on your Drive:
/content/drive/MyDrive/ImageNet1K
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0000.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0001.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0002.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0003.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0004.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0005.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0006.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0007.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0008.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0009.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0010.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0011.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0012.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0013.tar
/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0014.tar
/content/driv

In [ ]:
import os
import tarfile
import shutil

# Target directory structure
base_dir = "/content/imagenet-1k"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

source_folder = "/content/drive/MyDrive/ImageNet1K"
all_files = sorted(os.listdir(source_folder))
tar_files = [f for f in all_files if f.endswith('.tar')]

print(f"Found {len(tar_files)} tar archives in Google Drive.")

# Let's write a helper to safely extract tar files into structured directories
def extract_tar_to_imagenet(tar_path, target_root):
    print(f"Extracting {os.path.basename(tar_path)}...")
    with tarfile.open(tar_path, "r") as tar:
        # Check files inside the archive to match structure
        for member in tar.getmembers():
            if member.isfile():
                # Standard ImageNet archives contain paths like: train/n01440764/n01440764_10026.JPEG
                # Or flat files. We'll extract them preserving/creating class directories.
                name_parts = member.name.strip("/").split("/")

                if len(name_parts) >= 3 and name_parts[0] in ['train', 'val', 'validation']:
                    # Format matches: split/class/image.JPEG
                    split, category, img_name = name_parts[0], name_parts[1], name_parts[-1]
                    target_split = 'validation' if split in ['val', 'validation'] else 'train'
                    dest_folder = os.path.join(base_dir, target_split, category)
                else:
                    # If it's just class/image.JPEG or flat files, default to train folder under a structured category
                    category = name_parts[-2] if len(name_parts) > 1 else "unknown_class"
                    dest_folder = os.path.join(train_dir, category)

                os.makedirs(dest_folder, exist_ok=True)
                member.name = os.path.basename(member.name)
                tar.extract(member, path=dest_folder)

# For demonstration and system limits, let's extract the first few shards to build the structure.
# You can change the slice [:5] to extract all archives.
shards_to_extract = tar_files[:3]
for tar_file in shards_to_extract:
    full_path = os.path.join(source_folder, tar_file)
    try:
        extract_tar_to_imagenet(full_path, base_dir)
    except Exception as e:
        print(f"Error extracting {tar_file}: {e}")

# Show final structured directories
print("\nStructuring Check:")
for split in ['train', 'validation']:
    split_path = os.path.join(base_dir, split)
    if os.path.exists(split_path):
        classes = sorted(os.listdir(split_path))[:5]
        print(f"{split}/ contains {len(os.listdir(split_path))} classes. Examples: {classes}")

Found 130 tar archives in Google Drive.
Extracting imagenet1k-train-0000.tar...


/tmp/ipykernel_354/3956707016.py:42: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=dest_folder)


Extracting imagenet1k-train-0001.tar...
Error extracting imagenet1k-train-0001.tar: [Errno 107] Transport endpoint is not connected
Extracting imagenet1k-train-0002.tar...
Error extracting imagenet1k-train-0002.tar: [Errno 107] Transport endpoint is not connected: '/content/drive/MyDrive/ImageNet1K/imagenet1k-train-0002.tar'

Structuring Check:
train/ contains 1 classes. Examples: ['unknown_class']
validation/ contains 0 classes. Examples: []


In [ ]:
import os
import shutil
import tarfile

# Create local directories
local_tar_dir = "/content/temp_tars"
os.makedirs(local_tar_dir, exist_ok=True)

base_dir = "/content/imagenet-1k"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

source_folder = "/content/drive/MyDrive/ImageNet1K"

# Copying just 2 shards locally to test and structure them fast without timing out
shards_to_copy = ['imagenet1k-train-0000.tar', 'imagenet1k-train-0001.tar']

for shard in shards_to_copy:
    src_path = os.path.join(source_folder, shard)
    dst_path = os.path.join(local_tar_dir, shard)
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        print(f"Copying {shard} locally...")
        shutil.copy(src_path, dst_path)

print("Local copies completed. Checking archive internal structure...")

# Inspect the structure of the first archive
test_tar = os.path.join(local_tar_dir, 'imagenet1k-train-0000.tar')
if os.path.exists(test_tar):
    with tarfile.open(test_tar, 'r') as tar:
        members = tar.getmembers()
        print(f"Total items in {shards_to_copy[0]}: {len(members)}")
        print("Sample files inside archive:")
        for m in members[:10]:
            print(m.name)

Copying imagenet1k-train-0000.tar locally...
Copying imagenet1k-train-0001.tar locally...
Local copies completed. Checking archive internal structure...
Total items in imagenet1k-train-0000.tar: 3753
Sample files inside archive:
n02097047_2079.cls
n02097047_2079.jpg
n02097047_2079.json
n01682714_8546.cls
n01682714_8546.jpg
n01682714_8546.json
n03134739_3100.cls
n03134739_3100.jpg
n03134739_3100.json
n04254777_3064.cls


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import tarfile
import shutil
from sklearn.model_selection import train_test_split

# Configure paths
local_tar_dir = "/content/temp_tars"
base_dir = "/content/imagenet-1k"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

tar_files = [f for f in os.listdir(local_tar_dir) if f.endswith('.tar')]
print(f"Processing local tar files: {tar_files}")

# Group images by their class extracted from filenames
class_image_mapping = {}

for tar_name in tar_files:
    tar_path = os.path.join(local_tar_dir, tar_name)
    print(f"Scanning structure of {tar_name}...")
    with tarfile.open(tar_path, 'r') as tar:
        for member in tar.getmembers():
            if member.isfile() and member.name.lower().endswith(('.jpg', '.jpeg', '.png')):
                filename = os.path.basename(member.name)
                # Parse WordNet class ID (e.g., 'n02097047_2079.jpg' -> 'n02097047')
                class_id = filename.split('_')[0]
                if class_id not in class_image_mapping:
                    class_image_mapping[class_id] = []
                class_image_mapping[class_id].append((tar_path, member))

print(f"Found {len(class_image_mapping)} unique classes across shards.")

# Split and extract images into standard train/validation class subdirectories
for class_id, items in class_image_mapping.items():
    # Ensure we have class subfolders
    os.makedirs(os.path.join(train_dir, class_id), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_id), exist_ok=True)

    # If class has multiple images, split train/validation (80/20)
    if len(items) > 1:
        train_items, val_items = train_test_split(items, test_size=0.2, random_state=42)
    else:
        train_items, val_items = items, []

    # Extract to target destinations
    for tar_path, member in train_items:
        with tarfile.open(tar_path, 'r') as tar:
            member.name = os.path.basename(member.name)
            tar.extract(member, path=os.path.join(train_dir, class_id))

    for tar_path, member in val_items:
        with tarfile.open(tar_path, 'r') as tar:
            member.name = os.path.basename(member.name)
            tar.extract(member, path=os.path.join(val_dir, class_id))

# Validation check
print("\nDataset Structuring Completed Successfully!")
for split in ['train', 'validation']:
    split_path = os.path.join(base_dir, split)
    classes = sorted(os.listdir(split_path))
    print(f"{split_path}/ has {len(classes)} classes.")
    if classes:
        sample_class = classes[0]
        sample_class_path = os.path.join(split_path, sample_class)
        print(f"  -> Sample class '{sample_class}' contains: {os.listdir(sample_class_path)[:3]}")

Processing local tar files: ['imagenet1k-train-0001.tar', 'imagenet1k-train-0000.tar']
Scanning structure of imagenet1k-train-0001.tar...
Scanning structure of imagenet1k-train-0000.tar...
Found 918 unique classes across shards.


/tmp/ipykernel_354/3140932311.py:52: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(train_dir, class_id))
/tmp/ipykernel_354/3140932311.py:57: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(val_dir, class_id))



Dataset Structuring Completed Successfully!
/content/imagenet-1k/train/ has 919 classes.
  -> Sample class 'n01440764' contains: ['n01440764_13116.jpg']
/content/imagenet-1k/validation/ has 918 classes.
  -> Sample class 'n01440764' contains: ['n01440764_8528.jpg']


In [ ]:
import os

train_path = '/content/imagenet-1k/train'
val_path = '/content/imagenet-1k/validation'

def count_images(directory):
    if not os.path.exists(directory):
        return 0
    total_count = 0
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                total_count += 1
    return total_count

num_train_images = count_images(train_path)
num_val_images = count_images(val_path)

print(f"Total training images: {num_train_images}")
print(f"Total validation/test images: {num_val_images}")
print(f"Combined total images: {num_train_images + num_val_images}")

Total training images: 3001
Total validation/test images: 752
Combined total images: 3753


In [ ]:
import os

train_path = '/content/imagenet-1k/train'
val_path = '/content/imagenet-1k/validation'

def get_directory_size_gb(directory):
    if not os.path.exists(directory):
        return 0.0
    total_size_bytes = 0
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                total_size_bytes += os.path.getsize(file_path)
            except OSError:
                pass
    return total_size_bytes / (1024 ** 3)

train_size_gb = get_directory_size_gb(train_path)
val_size_gb = get_directory_size_gb(val_path)
total_size_gb = train_size_gb + val_size_gb

print(f"Training images size: {train_size_gb:.4f} GB")
print(f"Validation images size: {val_size_gb:.4f} GB")
print(f"Total dataset size: {total_size_gb:.4f} GB")

Training images size: 0.3323 GB
Validation images size: 0.0794 GB
Total dataset size: 0.4117 GB


In [ ]:
import os
import shutil
import tarfile
from sklearn.model_selection import train_test_split

# Configure paths
source_folder = "/content/drive/MyDrive/ImageNet1K"
local_tar_dir = "/content/temp_tars"
base_dir = "/content/imagenet-1k"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")

os.makedirs(local_tar_dir, exist_ok=True)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Find all shards in drive
all_drive_files = sorted(os.listdir(source_folder))
tar_files = [f for f in all_drive_files if f.endswith('.tar')]
print(f"Found {len(tar_files)} tar shards in Drive.")

# Loop over all shards to copy and extract dynamically to manage space and prevent disconnections
for index, shard in enumerate(tar_files):
    src_path = os.path.join(source_folder, shard)
    dst_path = os.path.join(local_tar_dir, shard)

    # 1. Copy shard locally (if not already done)
    if not os.path.exists(dst_path):
        print(f"[{index+1}/{len(tar_files)}] Copying {shard} locally...")
        try:
            shutil.copy(src_path, dst_path)
        except Exception as e:
            print(f"Error copying {shard}: {e}")
            continue

    # 2. Scan and extract directly to standard layout
    print(f"Extracting {shard}...")
    class_image_mapping = {}
    try:
        with tarfile.open(dst_path, 'r') as tar:
            for member in tar.getmembers():
                if member.isfile() and member.name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    filename = os.path.basename(member.name)
                    class_id = filename.split('_')[0]
                    if class_id not in class_image_mapping:
                        class_image_mapping[class_id] = []
                    class_image_mapping[class_id].append(member)

        # Split and save classes from this shard
        for class_id, members in class_image_mapping.items():
            os.makedirs(os.path.join(train_dir, class_id), exist_ok=True)
            os.makedirs(os.path.join(val_dir, class_id), exist_ok=True)

            if len(members) > 1:
                train_items, val_items = train_test_split(members, test_size=0.2, random_state=42)
            else:
                train_items, val_items = members, []

            with tarfile.open(dst_path, 'r') as tar:
                for member in train_items:
                    member.name = os.path.basename(member.name)
                    tar.extract(member, path=os.path.join(train_dir, class_id))
                for member in val_items:
                    member.name = os.path.basename(member.name)
                    tar.extract(member, path=os.path.join(val_dir, class_id))
    except Exception as e:
        print(f"Error processing {shard}: {e}")

    # 3. Clean up the local tar to free space
    if os.path.exists(dst_path):
        os.remove(dst_path)

print("\nAll available shards processed and structured successfully!")

Found 130 tar shards in Drive.
Extracting imagenet1k-train-0000.tar...


/tmp/ipykernel_354/2166817487.py:62: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(train_dir, class_id))
/tmp/ipykernel_354/2166817487.py:65: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(val_dir, class_id))


Extracting imagenet1k-train-0001.tar...
[3/130] Copying imagenet1k-train-0002.tar locally...
Extracting imagenet1k-train-0002.tar...
[4/130] Copying imagenet1k-train-0003.tar locally...
Extracting imagenet1k-train-0003.tar...
[5/130] Copying imagenet1k-train-0004.tar locally...
Extracting imagenet1k-train-0004.tar...
[6/130] Copying imagenet1k-train-0005.tar locally...
Extracting imagenet1k-train-0005.tar...
[7/130] Copying imagenet1k-train-0006.tar locally...
Extracting imagenet1k-train-0006.tar...
[8/130] Copying imagenet1k-train-0007.tar locally...
Extracting imagenet1k-train-0007.tar...
[9/130] Copying imagenet1k-train-0008.tar locally...
Extracting imagenet1k-train-0008.tar...
[10/130] Copying imagenet1k-train-0009.tar locally...
Extracting imagenet1k-train-0009.tar...
[11/130] Copying imagenet1k-train-0010.tar locally...
Extracting imagenet1k-train-0010.tar...
[12/130] Copying imagenet1k-train-0011.tar locally...
Extracting imagenet1k-train-0011.tar...
[13/130] Copying imagenet1k

In [ ]:
import os

base_dir = '/content/imagenet-1k'

print("Checking base directory:", os.listdir(base_dir) if os.path.exists(base_dir) else "Not Found")

for split in ['train', 'validation']:
    split_path = os.path.join(base_dir, split)
    if os.path.exists(split_path):
        subdirs = os.listdir(split_path)
        print(f"\n{split} folder contains {len(subdirs)} subdirectories (classes).")
        if subdirs:
            # Print a few samples
            print(f"Sample classes in {split}: {sorted(subdirs)[:10]}")

            # Count files in a sample class
            sample_class_path = os.path.join(split_path, subdirs[0])
            print(f"Files in class '{subdirs[0]}': {os.listdir(sample_class_path)[:5]}")

Checking base directory: Not Found


In [ ]:
import os

# Let's search recursively inside /content to find where 'imagenet-1k' or 'train' directories ended up
found_paths = []
for root, dirs, files in os.walk('/content'):
    # Stop deep traversal of git or config metadata to keep it quick
    if '.git' in root or '.config' in root:
         continue
    if 'imagenet-1k' in dirs or 'train' in dirs:
         found_paths.append(root)

print("Search Results inside /content:")
if found_paths:
    for path in set(found_paths):
        print(f"- Found relevant folders inside: {path}")
        try:
            print(f"  Contents: {os.listdir(path)[:10]}")
        except Exception:
            pass
else:
    print("Could not find any 'imagenet-1k' or 'train' directories in /content. Let's list everything in /content:")
    print(os.listdir('/content'))

Search Results inside /content:
Could not find any 'imagenet-1k' or 'train' directories in /content. Let's list everything in /content:
['.config', 'sample_data']


### Permanent Backup: Archive and Save Structured Dataset to Google Drive
This cell will run the extraction process and compress the structured `/content/imagenet-1k` directory into a single zip file saved at `/content/drive/MyDrive/imagenet1k_structured.zip`.

In [ ]:
import os
import shutil

# Source folder in Drive
drive_source = "/content/drive/MyDrive/ImageNet1K"
# Backup destination zip file
drive_backup_zip = "/content/drive/MyDrive/imagenet1k_structured.zip"

# We will compress the existing local folder once it's extracted
if os.path.exists("/content/imagenet-1k"):
    print("Archiving structured dataset to Google Drive for permanent storage...")
    shutil.make_archive("/content/drive/MyDrive/imagenet1k_structured", 'zip', "/content/imagenet-1k")
    print(f"Saved permanently to Google Drive at: {drive_backup_zip}")
else:
    print("Local structured folder /content/imagenet-1k not found. Please run the extraction shard script first, then run this to back it up permanently.")

Local structured folder /content/imagenet-1k not found. Please run the extraction shard script first, then run this to back it up permanently.


In [ ]:
import os
import shutil
import tarfile
import zipfile
from sklearn.model_selection import train_test_split

# Configure paths
source_folder = "/content/drive/MyDrive/ImageNet1K"
local_temp_extract = "/content/temp_extract"
drive_backup_zip = "/content/drive/MyDrive/imagenet1k_structured.zip"

os.makedirs(local_temp_extract, exist_ok=True)

# Find all shards in Google Drive
all_drive_files = sorted(os.listdir(source_folder))
tar_files = [f for f in all_drive_files if f.endswith('.tar')]
print(f"Found {len(tar_files)} tar shards in Drive.")

# Process and write directly to the persistent ZIP in Drive
for index, shard in enumerate(tar_files):
    src_path = os.path.join(source_folder, shard)
    local_shard_path = os.path.join("/content", shard)

    print(f"\n[{index+1}/{len(tar_files)}] Copying {shard} to local runtime...")
    try:
        shutil.copy(src_path, local_shard_path)
    except Exception as e:
        print(f"Error copying {shard}: {e}")
        continue

    print(f"Extracting and organizing {shard}...")
    class_image_mapping = {}
    try:
        with tarfile.open(local_shard_path, 'r') as tar:
            for member in tar.getmembers():
                if member.isfile() and member.name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    filename = os.path.basename(member.name)
                    class_id = filename.split('_')[0]
                    if class_id not in class_image_mapping:
                        class_image_mapping[class_id] = []
                    class_image_mapping[class_id].append(member)

        # Extract files locally to temporary path
        for class_id, members in class_image_mapping.items():
            if len(members) > 1:
                train_items, val_items = train_test_split(members, test_size=0.2, random_state=42)
            else:
                train_items, val_items = members, []

            with tarfile.open(local_shard_path, 'r') as tar:
                # Save train images
                for member in train_items:
                    member.name = os.path.basename(member.name)
                    dest_dir = os.path.join(local_temp_extract, "train", class_id)
                    os.makedirs(dest_dir, exist_ok=True)
                    tar.extract(member, path=dest_dir)

                # Save validation images
                for member in val_items:
                    member.name = os.path.basename(member.name)
                    dest_dir = os.path.join(local_temp_extract, "validation", class_id)
                    os.makedirs(dest_dir, exist_ok=True)
                    tar.extract(member, path=dest_dir)

        # Append the extracted local files to the persistent Google Drive ZIP archive
        print(f"Appending {shard} to permanent Google Drive ZIP archive...")
        with zipfile.ZipFile(drive_backup_zip, 'a', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(local_temp_extract):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, local_temp_extract)
                    zipf.write(file_path, arcname)

        # Clean up temporary folders and copied shard to free local space
        shutil.rmtree(local_temp_extract)
        os.makedirs(local_temp_extract, exist_ok=True)
        if os.path.exists(local_shard_path):
            os.remove(local_shard_path)

        print(f"Successfully backed up and cleaned {shard}!")

    except Exception as e:
        print(f"Error processing {shard}: {e}")

print("\nAll shards have been processed and permanently stored in your Google Drive ZIP file!")

Found 130 tar shards in Drive.

[1/130] Copying imagenet1k-train-0000.tar to local runtime...
Extracting and organizing imagenet1k-train-0000.tar...


/tmp/ipykernel_353/1054508305.py:56: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=dest_dir)
/tmp/ipykernel_353/1054508305.py:63: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=dest_dir)


Appending imagenet1k-train-0000.tar to permanent Google Drive ZIP archive...


KeyboardInterrupt: 

### Option 1: Persistent Extraction Directly to Google Drive
This cell reads your `.tar` shards from `/content/drive/MyDrive/ImageNet1K` and extracts the train/validation images directly into structured class directories at `/content/drive/MyDrive/imagenet-1k`.

In [ ]:
import os
import time
import shutil
import tarfile
from google.colab import drive
from sklearn.model_selection import train_test_split

# Configure persistent Google Drive paths
source_folder = "/content/drive/MyDrive/ImageNet1K"
persistent_dataset_root = "/content/drive/MyDrive/imagenet-1k"
train_dir = os.path.join(persistent_dataset_root, "train")
val_dir = os.path.join(persistent_dataset_root, "validation")

def check_and_remount():
    """Checks if Google Drive is connected, and remounts if disconnected."""
    for attempt in range(5):
        if os.path.exists(source_folder):
            return True
        print(f"Google Drive disconnected! Attempting to remount (Attempt {attempt+1}/5)... ")
        try:
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
        except Exception as e:
            print(f"Remount failed: {e}")
            time.sleep(10)
    return False

if not check_and_remount():
    raise RuntimeError("Could not maintain Google Drive connection. Please manually run a drive mount cell.")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Find all shards in Google Drive
all_drive_files = sorted(os.listdir(source_folder))
tar_files = [f for f in all_drive_files if f.endswith('.tar')]
print(f"Found {len(tar_files)} tar shards in Drive.")

# Process shards dynamically
for index, shard in enumerate(tar_files):
    src_path = os.path.join(source_folder, shard)
    local_shard_path = os.path.join("/content", shard)

    print(f"\n[{index+1}/{len(tar_files)}] Copying {shard} locally...")

    # Safe copy with retry and remount logic
    copied = False
    for retry in range(3):
        if not check_and_remount():
            break
        try:
            shutil.copy(src_path, local_shard_path)
            copied = True
            break
        except Exception as e:
            print(f"Copy failed (Retry {retry+1}/3): {e}. Waiting to retry...")
            time.sleep(15)

    if not copied:
        print(f"Skipping {shard} due to persistent connection issues.")
        continue

    print(f"Extracting and organizing {shard} directly to Google Drive...")
    class_image_mapping = {}
    try:
        with tarfile.open(local_shard_path, 'r') as tar:
            for member in tar.getmembers():
                if member.isfile() and member.name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    filename = os.path.basename(member.name)
                    class_id = filename.split('_')[0]
                    if class_id not in class_image_mapping:
                        class_image_mapping[class_id] = []
                    class_image_mapping[class_id].append(member)

        # Extract directly to target Drive folders
        for class_id, members in class_image_mapping.items():
            if len(members) > 1:
                train_items, val_items = train_test_split(members, test_size=0.2, random_state=42)
            else:
                train_items, val_items = members, []

            # Skip extracting if all target files for this class already exist
            need_extract_train = [m for m in train_items if not os.path.exists(os.path.join(train_dir, class_id, os.path.basename(m.name)))]
            need_extract_val = [m for m in val_items if not os.path.exists(os.path.join(val_dir, class_id, os.path.basename(m.name)))]

            if need_extract_train or need_extract_val:
                os.makedirs(os.path.join(train_dir, class_id), exist_ok=True)
                os.makedirs(os.path.join(val_dir, class_id), exist_ok=True)

                with tarfile.open(local_shard_path, 'r') as tar:
                    for member in need_extract_train:
                        member.name = os.path.basename(member.name)
                        tar.extract(member, path=os.path.join(train_dir, class_id))
                    for member in need_extract_val:
                        member.name = os.path.basename(member.name)
                        tar.extract(member, path=os.path.join(val_dir, class_id))

        # Clean up local shard copy to free up local environment disk
        if os.path.exists(local_shard_path):
            os.remove(local_shard_path)

        print(f"Completed and saved {shard} directly to your Google Drive!")
        # Sleep slightly between shards to avoid API throttling
        time.sleep(3)

    except Exception as e:
        print(f"Error processing {shard}: {e}")

print("\nAll ImageNet-1K shards are extracted and saved permanently in your Google Drive!")

Found 130 tar shards in Drive.

[1/130] Copying imagenet1k-train-0000.tar locally...
Extracting and organizing imagenet1k-train-0000.tar directly to Google Drive...
Completed and saved imagenet1k-train-0000.tar directly to your Google Drive!

[2/130] Copying imagenet1k-train-0001.tar locally...
Extracting and organizing imagenet1k-train-0001.tar directly to Google Drive...


/tmp/ipykernel_353/2541276813.py:93: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(train_dir, class_id))
/tmp/ipykernel_353/2541276813.py:96: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(val_dir, class_id))


Completed and saved imagenet1k-train-0001.tar directly to your Google Drive!

[3/130] Copying imagenet1k-train-0002.tar locally...
Extracting and organizing imagenet1k-train-0002.tar directly to Google Drive...
Completed and saved imagenet1k-train-0002.tar directly to your Google Drive!

[4/130] Copying imagenet1k-train-0003.tar locally...
Extracting and organizing imagenet1k-train-0003.tar directly to Google Drive...
Completed and saved imagenet1k-train-0003.tar directly to your Google Drive!

[5/130] Copying imagenet1k-train-0004.tar locally...
Extracting and organizing imagenet1k-train-0004.tar directly to Google Drive...
Completed and saved imagenet1k-train-0004.tar directly to your Google Drive!

[6/130] Copying imagenet1k-train-0005.tar locally...
Extracting and organizing imagenet1k-train-0005.tar directly to Google Drive...
Completed and saved imagenet1k-train-0005.tar directly to your Google Drive!

[7/130] Copying imagenet1k-train-0006.tar locally...
Extracting and organizing

/tmp/ipykernel_353/2541276813.py:93: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(train_dir, class_id))
/tmp/ipykernel_353/2541276813.py:96: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=os.path.join(val_dir, class_id))


Completed and saved imagenet1k-train-0053.tar directly to your Google Drive!

[55/130] Copying imagenet1k-train-0054.tar locally...
Extracting and organizing imagenet1k-train-0054.tar directly to Google Drive...
Completed and saved imagenet1k-train-0054.tar directly to your Google Drive!

[56/130] Copying imagenet1k-train-0055.tar locally...
Extracting and organizing imagenet1k-train-0055.tar directly to Google Drive...
Completed and saved imagenet1k-train-0055.tar directly to your Google Drive!

[57/130] Copying imagenet1k-train-0056.tar locally...
Extracting and organizing imagenet1k-train-0056.tar directly to Google Drive...
Completed and saved imagenet1k-train-0056.tar directly to your Google Drive!

[58/130] Copying imagenet1k-train-0057.tar locally...
Extracting and organizing imagenet1k-train-0057.tar directly to Google Drive...
Completed and saved imagenet1k-train-0057.tar directly to your Google Drive!

[59/130] Copying imagenet1k-train-0058.tar locally...
Extracting and organ

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/imagenet-1k"

TRAIN_PATH = f"{DATASET_PATH}/train"
VAL_PATH = f"{DATASET_PATH}/validation"

import os

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Folders:", os.listdir(DATASET_PATH))
print("Validation classes:", len(os.listdir(VAL_PATH)))

Dataset exists: True
Folders: ['train', 'validation']
Validation classes: 1000


In [ ]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder(VAL_PATH)

print("Number of classes:", len(dataset.classes))
print("Number of images:", len(dataset))

Number of classes: 1000
Number of images: 21293


In [ ]:
%cd /content/ml-fastvit

/content/ml-fastvit


In [ ]:
!pip install -q timm==0.9.16 scipy

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4


In [ ]:
!mkdir -p checkpoints

!wget -q --show-progress \
https://docs-assets.developer.apple.com/ml-research/models/fastvit/image_classification_models/fastvit_t8_reparam.pth.tar \
-O checkpoints/fastvit_t8_reparam.pth.tar

checkpoints/fastvit 100%[===================>]  15.31M  --.-KB/s    in 0.08s   


In [ ]:
!python validate.py "{DATASET_PATH}" \
    --model fastvit_t8 \
    --checkpoint checkpoints/fastvit_t8_reparam.pth.tar \
    --use-inference-mode \
    --split validation \
    --input-size 3 256 256 \
    --batch-size 64 \
    --workers 2 \
    --log-freq 10 \
    --results-file "/content/drive/MyDrive/fastvit_results.csv"

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
If for semantic segmentation, please install mmsegmentation first
If for detection, please install mmdetection first
/content/ml-fastvit/models/fastvit.py:956: UserWarning: Overwriting fastvit_t8 in registry with models.fastvit.fastvit_t8. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/ml-fastvit/models/fastvit.py:978: UserWarning: Overwriting fastvit_t12 in regi